In [ ]:
import pandas as pd


baseline = pd.read_csv(
    "../results/baseline_results.csv"
)

rag = pd.read_csv(
    "../results/rag_results.csv"
)

sce = pd.read_csv(
    "../results/sce_results.csv"
)


print(len(baseline))
print(len(rag))
print(len(sce))

In [ ]:
baseline["answer_length"] = (
    baseline["generated_answer"]
    .str.len()
)

rag["answer_length"] = (
    rag["generated_answer"]
    .str.len()
)


print(
    baseline["answer_length"].describe()
)

print(
    rag["answer_length"].describe()
)

In [ ]:
sce["semantic_similarity_score"].describe()

In [ ]:
hallucination_rate = (
    sce["hallucination_flag"].mean()
    * 100
)

print(
    f"Hallucination rate: {hallucination_rate:.2f}%"
)

In [ ]:
results_summary = pd.DataFrame({

    "Approach": [
        "Baseline LLM",
        "RAG",
        "RAG + SCE"
    ],

    "Samples": [
        len(baseline),
        len(rag),
        len(sce)
    ],

    "Average Semantic Score": [
        None,
        None,
        sce["semantic_similarity_score"].mean()
    ],

    "Hallucination Rate (%)": [
        None,
        None,
        sce["hallucination_flag"].mean()*100
    ]

})


results_summary

In [ ]:
# from bert_score import score


# P, R, F1 = score(
#     rag["generated_answer"].tolist(),
#     rag["best_answer"].tolist(),
#     lang="en"
# )


# print(F1.mean())

Transformer limmitation here, will have to downgrade, changes project structure

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim


embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


scores = []


for answer, reference in zip(
    rag["generated_answer"],
    rag["best_answer"]
):

    answer_embedding = embedding_model.encode(
        answer,
        convert_to_tensor=True
    )

    reference_embedding = embedding_model.encode(
        reference,
        convert_to_tensor=True
    )


    similarity = cos_sim(
        answer_embedding,
        reference_embedding
    )


    scores.append(
        float(similarity[0][0])
    )


print(sum(scores)/len(scores))

In [ ]:
# import evaluate


# bertscore = evaluate.load("bertscore")

# # 
# results = bertscore.compute(
#     predictions=rag["generated_answer"].tolist(),
#     references=rag["best_answer"].tolist(),
#     lang="en"
# )


# print(
#     sum(results["f1"]) / len(results["f1"])
# )

In [ ]:
from rouge_score import rouge_scorer


scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)


scores = []


for answer, reference in zip(
    rag["generated_answer"],
    rag["best_answer"]
):

    result = scorer.score(
        reference,
        answer
    )

    scores.append(
        result["rougeL"].fmeasure
    )


print(sum(scores)/len(scores))

In [ ]:
import matplotlib.pyplot as plt


plt.hist(
    sce["semantic_similarity_score"],
    bins=20
)

plt.xlabel(
    "Semantic Consistency Score"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Distribution of SCE Scores"
)

plt.show()